# Validación offline de librería Python

Este notebook valida un wheel local sin usar Internet.

In [1]:
from pathlib import Path
import json
import os
import sys
import zipfile
from email.parser import Parser

def resolve_model_path():
    configured = os.environ.get('MODEL_PATH', '').strip()
    if configured:
        return Path(configured).expanduser().resolve()
    if os.environ.get('DATABRICKS_RUNTIME_VERSION'):
        try:
            dbutils.widgets.text('model_path', '')
            configured = dbutils.widgets.get('model_path').strip()
        except Exception:
            configured = ''
        if configured:
            return Path(configured).resolve()
        raise RuntimeError('En Databricks indique MODEL_PATH o el widget model_path con una ruta /Volumes/...')
    working_directory = Path.cwd().resolve()
    if (working_directory / 'model-metadata.json').is_file():
        return working_directory
    relative_model = Path('models') / 'libs' / '.sentence-transformers.partial-e23af438a9814ec4b466c80f90d13089'
    for root in (working_directory, *working_directory.parents):
        candidate = root / relative_model
        if (candidate / 'model-metadata.json').is_file():
            return candidate.resolve()
    raise RuntimeError('No se encontró la carpeta de la librería. Ejecute desde el repositorio o defina MODEL_PATH.')

MODEL_PATH = resolve_model_path()
metadata = json.loads((MODEL_PATH / 'model-metadata.json').read_text(encoding='utf-8'))
assert metadata['model_type'] == 'lib', 'Tipo de artefacto incorrecto'
wheel = MODEL_PATH / metadata['wheel_file']
assert wheel.is_file(), f'Wheel faltante: {wheel}'
LFS_PREFIX = b'version https://git-lfs.github.com/spec/v1'
with wheel.open('rb') as handle:
    assert not handle.read(len(LFS_PREFIX)).startswith(LFS_PREFIX), 'Puntero Git LFS detectado; ejecute git lfs pull.'
print(f'Library path: {MODEL_PATH}')
print(f'wheel: {wheel.name}')
print(f'python: {sys.version.split()[0]}')

Library path: C:\Users\tarug\Desktop\Databricks Offline\models\libs\.sentence-transformers.partial-e23af438a9814ec4b466c80f90d13089
wheel: sentence_transformers-4.0.1-py3-none-any.whl
python: 3.12.3


In [2]:
with zipfile.ZipFile(wheel) as archive:
    metadata_members = [name for name in archive.namelist() if name.endswith('.dist-info/METADATA')]
    assert len(metadata_members) == 1, f'METADATA ambiguo o faltante: {metadata_members}'
    wheel_metadata = Parser().parsestr(archive.read(metadata_members[0]).decode('utf-8'))
assert wheel_metadata['Name'] == metadata['name'], 'Name del wheel no coincide con metadata'
assert wheel_metadata['Version'] == metadata['revision'], 'Version del wheel no coincide con metadata'
print(f"Wheel METADATA: Name={wheel_metadata['Name']} Version={wheel_metadata['Version']}")

Wheel METADATA: Name=sentence-transformers Version=4.0.1


In [3]:
import importlib
import importlib.metadata
import subprocess
import tempfile

with tempfile.TemporaryDirectory(prefix='offline-lib-validation-') as temporary:
    target = Path(temporary) / 'site-packages'
    subprocess.run([sys.executable, '-m', 'pip', 'install', str(wheel), '--no-deps', '--target', str(target)], check=True)
    sys.path.insert(0, str(target))
    try:
        importlib.invalidate_caches()
        for module_name in list(sys.modules):
            if module_name == metadata['import_name'] or module_name.startswith(metadata['import_name'] + '.'):
                del sys.modules[module_name]
        module = __import__(metadata['import_name'])
        distributions = importlib.metadata.distributions(path=[str(target)])
        installed_version = next(
            distribution.version for distribution in distributions
            if distribution.metadata['Name'] == metadata['name']
        )
        assert installed_version == metadata['revision'], f'Versión instalada inesperada: {installed_version}'
        print(f"import: {metadata['import_name']} {installed_version}")
    finally:
        sys.path.remove(str(target))

import: sentence_transformers 4.0.1


In [4]:
print('VALIDATION OK')
print('Libreria instalada y verificada usando unicamente archivos locales.')

VALIDATION OK
Libreria instalada y verificada usando unicamente archivos locales.
